# Stable Tiny Overfit Check

Короткий Colab-ноутбук для проверки training pipeline курсовой.

Он специально обучает модель на тех же 16/32 клипах, на которых затем измеряет accuracy. Это не итоговое качество модели, а проверка того, что загрузка видео, labels, loss, backward и optimizer работают.

Запускай ячейки строго сверху вниз. В конце будет один из результатов:

```text
PIPELINE PASS  -> можно переходить к полноценным моделям
PIPELINE CHECK -> сначала разбираем сохранённые ошибки
```

## 1. Подключить Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Найти и распаковать датасет

На Google Drive должен лежать `kaggle_lipreading_dataset.zip` или `colab_lipreading_dataset.zip`. Если датасет уже распакован в текущей Colab-сессии, повторная распаковка не выполняется.

In [ ]:
import zipfile
from pathlib import Path
import shutil

extract_dir = Path("/content/lipreading_data_fixed")
expected_dir = extract_dir / "ru_dataset" / "mouth_crops_padded"

if expected_dir.exists():
    print("Dataset already exists:", extract_dir)
else:
    print("Dataset not found, searching zip on Google Drive...")

    zip_names = ["kaggle_lipreading_dataset.zip", "colab_lipreading_dataset.zip"]
    matches = []
    for zip_name in zip_names:
        matches.extend(Path("/content/drive").rglob(zip_name))

    print("matches:", matches)

    if len(matches) == 0:
        raise FileNotFoundError("Upload kaggle_lipreading_dataset.zip or colab_lipreading_dataset.zip to Google Drive first")

    zip_path = matches[0]
    print("zip:", zip_path)

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        for member in zip_ref.infolist():
            fixed_name = member.filename.replace("\\", "/")
            target_path = extract_dir / fixed_name

            if fixed_name.endswith("/"):
                target_path.mkdir(parents=True, exist_ok=True)
                continue

            target_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_ref.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())

    print("Dataset restored:", extract_dir)

data_root = extract_dir / "ru_dataset"
print("data_root:", data_root)
print("mouth crops exist:", (data_root / "mouth_crops_padded").exists())

## 3. Импорты, таблицы и пути

In [ ]:
from pathlib import Path
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'

split_name = 'ml_splits_mouth_crops_padded_clean_manual_no_spk09_10'
split_dir = data_root / split_name / 'speaker_top10'

if not (split_dir / 'train.csv').exists():
    split_name = 'ml_splits_mouth_crops_padded_no_spk09_10'
    split_dir = data_root / split_name / 'speaker_top10'

if not (split_dir / 'train.csv').exists():
    raise FileNotFoundError(f'Train split was not found under {data_root}')

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
test_df = pd.read_csv(split_dir / 'test.csv')

vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

checkpoint_dir = Path('/content/drive/MyDrive/lipreading_sem4')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print('device:', device)
print('split_dir:', split_dir)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('vocab:', vocab)
print('results will be saved to:', checkpoint_dir)

## 4. Загрузка одного видеоклипа

Каждый mp4 преобразуется в grayscale-тензор `[1, 24, 96, 96]`: канал, кадры, высота, ширина.

In [ ]:
def load_video(path, num_frames=24, size=96, grayscale=False):
    cap = cv2.VideoCapture(str(path))

    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')

    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    for frame_idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()

        if not ok:
            continue

        if grayscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            frame = cv2.resize(frame, (size, size))
            frame = frame[:, :, None]
        else:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (size, size))

        frame = frame / 255.0
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError(f'No frames loaded: {path}')

    while len(frames) < num_frames:
        frames.append(frames[-1])

    video = np.stack(frames)
    video = torch.tensor(video, dtype=torch.float32)
    video = video.permute(3, 0, 1, 2)

    return video

## 5. Один шаг обучения и оценка

`train_one_epoch` обновляет веса. `evaluate` только считает loss и accuracy.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)

            total_loss += loss.item() * batch_x.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)

    return total_loss / total, correct / total

## 6. Стабильный tiny-overfit

Видео кэшируются в памяти. Проверяются пути, labels, дубликаты, движение между кадрами и дисперсия пикселей. Диагностическая модель использует GroupNorm, поэтому результат не зависит от статистик маленького batch.

In [ ]:
import hashlib
import random


def set_reproducible_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def make_round_robin_tiny_df(df, n_examples=16, random_state=42):
    rng = np.random.default_rng(random_state)
    groups = []

    for word in sorted(df['word'].unique()):
        group = df[df['word'] == word].copy().reset_index(drop=True)
        order = rng.permutation(len(group))
        groups.append(group.iloc[order].reset_index(drop=True))

    rows = []
    position = 0
    while len(rows) < n_examples:
        added = False
        for group in groups:
            if position < len(group):
                rows.append(group.iloc[position])
                added = True
                if len(rows) >= n_examples:
                    break
        if not added:
            break
        position += 1

    return pd.DataFrame(rows).sample(
        frac=1.0,
        random_state=random_state,
    ).reset_index(drop=True)


class CachedTinyDataset(Dataset):
    """Decode every tiny-test clip once, then reuse the same tensors."""

    def __init__(self, df, data_root, word_to_idx, idx_to_word):
        self.df = df.reset_index(drop=True).copy()
        videos = []
        labels = []
        fingerprints = []
        motion_scores = []
        pixel_stds = []
        round_trip_ok = []

        for _, row in self.df.iterrows():
            video_path = data_root / row['clip_path']
            if not video_path.exists():
                raise FileNotFoundError(f'Tiny-overfit clip does not exist: {video_path}')

            video = load_video(video_path, grayscale=True)
            label = word_to_idx[row['word']]

            videos.append(video)
            labels.append(label)
            fingerprints.append(hashlib.sha256(video.numpy().tobytes()).hexdigest())
            motion_scores.append((video[:, 1:] - video[:, :-1]).abs().mean().item())
            pixel_stds.append(video.std().item())
            round_trip_ok.append(idx_to_word[label] == row['word'])

        self.videos = torch.stack(videos)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.df['class_id'] = labels
        self.df['round_trip_ok'] = round_trip_ok
        self.df['tensor_fingerprint'] = fingerprints
        self.df['motion_score'] = motion_scores
        self.df['pixel_std'] = pixel_stds

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]


def audit_cached_tiny_dataset(dataset):
    audit_df = dataset.df.copy()
    conflicting_hashes = (
        audit_df.groupby('tensor_fingerprint')['class_id']
        .nunique()
        .loc[lambda counts: counts > 1]
        .index
        .tolist()
    )
    zero_motion_count = int((audit_df['motion_score'] <= 0.0).sum())
    zero_variance_count = int((audit_df['pixel_std'] <= 0.0).sum())
    round_trip_ok = bool(audit_df['round_trip_ok'].all())

    print('cached tensor shape:', dataset.videos.shape)
    print('label round trips valid:', round_trip_ok)
    print('conflicting duplicate tensors:', len(conflicting_hashes))
    print('zero-motion clips:', zero_motion_count)
    print('zero-variance clips:', zero_variance_count)
    print('motion score min/mean:', audit_df['motion_score'].min(), audit_df['motion_score'].mean())
    display(
        audit_df[
            ['clip_path', 'word', 'speaker_id', 'class_id', 'motion_score', 'pixel_std']
        ]
    )

    data_checks_passed = (
        round_trip_ok
        and not conflicting_hashes
        and zero_motion_count == 0
        and zero_variance_count == 0
    )
    return audit_df, data_checks_passed


class TinyFrameCNNBiGRU(nn.Module):
    """Small deterministic diagnostic model with batch-independent normalization."""

    def __init__(self, num_classes, cnn_dim=64, gru_hidden=64):
        super().__init__()
        self.frame_encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.GroupNorm(4, 32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, cnn_dim, kernel_size=3, padding=1),
            nn.GroupNorm(8, cnn_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.temporal_encoder = nn.GRU(
            input_size=cnn_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, x):
        batch_size, channels, frames, height, width = x.shape
        x = x.permute(0, 2, 1, 3, 4)
        x = x.reshape(batch_size * frames, channels, height, width)
        frame_features = self.frame_encoder(x)
        frame_features = frame_features.reshape(batch_size, frames, -1)
        _, hidden = self.temporal_encoder(frame_features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)


def collect_tiny_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            outputs = model(batch_x.to(device))
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(batch_y.tolist())
    return all_preds, all_labels


def show_tiny_error_triplets(dataset, error_df, output_path):
    if error_df.empty:
        print('No remaining tiny-overfit errors.')
        return None

    n_rows = len(error_df)
    fig, axes = plt.subplots(n_rows, 3, figsize=(9, 2.8 * n_rows), squeeze=False)

    for row_number, error in error_df.reset_index(drop=True).iterrows():
        dataset_index = int(error['dataset_index'])
        video = dataset.videos[dataset_index][0]
        frame_ids = [0, video.shape[0] // 2, video.shape[0] - 1]
        titles = ['first frame', 'middle frame', 'last frame']

        for column, (frame_id, title) in enumerate(zip(frame_ids, titles)):
            axes[row_number, column].imshow(video[frame_id].numpy(), cmap='gray', vmin=0, vmax=1)
            axes[row_number, column].axis('off')
            axes[row_number, column].set_title(
                f"{title}\ntrue={error['true_word']} pred={error['pred_word']}"
            )

    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    print('saved error triplets:', output_path)
    return output_path


def run_stable_tiny_overfit_test(
    n_examples,
    pass_threshold,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
):
    set_reproducible_seed(seed)
    tiny_df = make_round_robin_tiny_df(
        train_df,
        n_examples=n_examples,
        random_state=seed + n_examples,
    )
    print('tiny subset size:', len(tiny_df))
    print('word counts:')
    print(tiny_df['word'].value_counts())
    print('speakers:', sorted(tiny_df['speaker_id'].unique()))

    tiny_dataset = CachedTinyDataset(tiny_df, data_root, word_to_idx, idx_to_word)
    audit_df, data_checks_passed = audit_cached_tiny_dataset(tiny_dataset)

    generator = torch.Generator().manual_seed(seed)
    tiny_train_loader = DataLoader(
        tiny_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    tiny_eval_loader = DataLoader(
        tiny_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    tiny_model = TinyFrameCNNBiGRU(num_classes=len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        tiny_model.parameters(),
        lr=learning_rate,
        weight_decay=0.0,
    )

    history_rows = []
    best_memorization_acc = -1.0
    best_eval_loss = float('inf')
    checkpoint_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_best.pt'

    for epoch in range(1, num_epochs + 1):
        train_loss, train_step_acc = train_one_epoch(
            tiny_model,
            tiny_train_loader,
            criterion,
            optimizer,
            device,
        )
        eval_loss, memorization_acc = evaluate(
            tiny_model,
            tiny_eval_loader,
            criterion,
            device,
        )
        history_rows.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_step_acc': train_step_acc,
            'eval_loss_on_same_clips': eval_loss,
            'memorization_acc': memorization_acc,
        })

        improved = (
            memorization_acc > best_memorization_acc
            or (
                memorization_acc == best_memorization_acc
                and eval_loss < best_eval_loss
            )
        )
        if improved:
            best_memorization_acc = memorization_acc
            best_eval_loss = eval_loss
            torch.save(
                {
                    'epoch': epoch,
                    'model_state_dict': tiny_model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'vocab': vocab,
                    'word_to_idx': word_to_idx,
                    'idx_to_word': idx_to_word,
                    'best_memorization_acc': best_memorization_acc,
                    'best_eval_loss': best_eval_loss,
                    'n_examples': n_examples,
                    'seed': seed,
                },
                checkpoint_path,
            )

        if epoch == 1 or epoch % 10 == 0 or memorization_acc >= 1.0:
            print(
                f'tiny {n_examples} | epoch {epoch}/{num_epochs} | '
                f'train_loss={train_loss:.4f} | same-clips loss={eval_loss:.4f} | '
                f'memorization_acc={memorization_acc:.4f} | best={best_memorization_acc:.4f}'
            )

        if memorization_acc >= 1.0:
            print('early stop: perfect memorization reached')
            break

    history_df = pd.DataFrame(history_rows)
    history_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_history.csv'
    history_df.to_csv(history_path, index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    tiny_model.load_state_dict(checkpoint['model_state_dict'])
    preds, labels = collect_tiny_predictions(tiny_model, tiny_eval_loader, device)

    error_rows = []
    for dataset_index, (pred, label) in enumerate(zip(preds, labels)):
        if pred == label:
            continue
        metadata = audit_df.iloc[dataset_index]
        error_rows.append({
            'dataset_index': dataset_index,
            'clip_path': metadata['clip_path'],
            'speaker_id': metadata['speaker_id'],
            'true_id': label,
            'pred_id': pred,
            'true_word': idx_to_word[label],
            'pred_word': idx_to_word[pred],
            'motion_score': metadata['motion_score'],
            'pixel_std': metadata['pixel_std'],
        })

    error_columns = [
        'dataset_index', 'clip_path', 'speaker_id', 'true_id', 'pred_id',
        'true_word', 'pred_word', 'motion_score', 'pixel_std',
    ]
    errors_df = pd.DataFrame(error_rows, columns=error_columns)
    errors_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_errors.csv'
    errors_df.to_csv(errors_path, index=False)

    triplets_path = checkpoint_dir / f'stable_tiny_overfit_{n_examples}_error_triplets.png'
    show_tiny_error_triplets(tiny_dataset, errors_df, triplets_path)

    passed = data_checks_passed and best_memorization_acc >= pass_threshold
    verdict = (
        f'PASS: best memorization accuracy {best_memorization_acc:.1%} '
        f'reached the {pass_threshold:.0%} threshold.'
        if passed
        else (
            f'CHECK: best memorization accuracy {best_memorization_acc:.1%}; '
            f'required {pass_threshold:.0%}. Inspect the saved errors and data checks.'
        )
    )
    print(verdict)
    print('saved history:', history_path)
    print('saved errors:', errors_path)

    return {
        'n_examples': n_examples,
        'model': tiny_model,
        'history': history_df,
        'best_memorization_acc': best_memorization_acc,
        'pass_threshold': pass_threshold,
        'data_checks_passed': data_checks_passed,
        'passed': passed,
        'checkpoint_path': checkpoint_path,
        'history_path': history_path,
        'errors_path': errors_path,
        'error_triplets_path': triplets_path if not errors_df.empty else None,
        'errors': errors_df,
        'verdict': verdict,
    }

## 7. Проверка на 16 клипах

Порог PASS: не меньше 95%. Обучение остановится раньше 150 эпох, если будет достигнуто 100%.

In [ ]:
tiny16_result = run_stable_tiny_overfit_test(
    n_examples=16,
    pass_threshold=0.95,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
)

## 8. Проверка на 32 клипах

Порог PASS: не меньше 90%.

In [ ]:
tiny32_result = run_stable_tiny_overfit_test(
    n_examples=32,
    pass_threshold=0.90,
    num_epochs=150,
    batch_size=4,
    learning_rate=3e-3,
    seed=42,
)

## 9. Итоговое решение

In [ ]:
tiny_overfit_summary = pd.DataFrame([
    {
        'n_examples': result['n_examples'],
        'best_memorization_acc': result['best_memorization_acc'],
        'required_acc': result['pass_threshold'],
        'data_checks_passed': result['data_checks_passed'],
        'passed': result['passed'],
        'epochs_ran': len(result['history']),
        'checkpoint_path': str(result['checkpoint_path']),
        'errors_path': str(result['errors_path']),
    }
    for result in [tiny16_result, tiny32_result]
])

display(tiny_overfit_summary)

tiny_pipeline_pass = bool(tiny_overfit_summary['passed'].all())
if tiny_pipeline_pass:
    print('PIPELINE PASS: both tiny-overfit checks passed. Full-dataset experiments may continue.')
else:
    print('PIPELINE CHECK: stop here. Do not run the heavy model cells yet.')

## Что прислать после запуска

Сохрани или скачай выполненный `.ipynb` и пришли его сюда. Также достаточно прислать таблицу `tiny_overfit_summary` и последнюю строку `PIPELINE PASS` / `PIPELINE CHECK`.

CSV, checkpoints и изображения оставшихся ошибок сохраняются в:

```text
/content/drive/MyDrive/lipreading_sem4
```